# How To Leverage OpenAI Training Budget ($$$) For Free!!!

## Why?

Due to the cost of training a `GPTModel`.

To put the scale of training budget into perspective, consider OpenAI's smallest `GPT-2` model containing `124M` parameters.

The original `GPT-2` model was trained on `32 TPU v3 chips` for approximately 7 days (168 hours). Assuming a cloud cost of roughly `$8 per TPU-hour`, the estimated training cost was around `$43,000`.

Fortunately, today we do not need to spend tens of thousands of dollars to obtain these learned parameters. 

## How?
Previously, we trained our `GPTModel` model using a limited dataset comprising a short-story book. This approach allowed us to focus on the fundamentals without the need for extensive time and computational resources.

OpenAI has publicly released the pretrained `GPT-2` parameters, allowing us to load them directly into our own `GPTModel` implementation and immediately benefit from OpenAI's substantial training investment.

So, let’s load these parameters into our `GPTModel` implementation and use the model for text generation. 

Here, parameters refer to the weight & biases stored in the `.weight` & `bias` attributes of PyTorch’s Linear and Embedding layers.

## Goal:
> "Given the input sequence `"Every effort moves you"` generate the next tokens in the sequence autoregressively using OpenAI `GPT-2` model parameters."

## Recipe:
1. Install dependencies to download OpenAI trained GPT-2 model weights.
2. Download OpenAI trained GPT-2 model weights.
3. Write a mapping layer from params dictionary to the appropriate GPTModel components.
4. Import our already codified `GPTModel` implementation.
5. Initialize our `GPTModel` implementation (by default with random weights).
6. Load OpenAI trained `GPT-2` model parameters from OpenAI into our very own implementation `GPTModel`.
7. Import sophisticated text generation app function which we discussed in the last [computational essay](https://github.com/umairkhancis/llm-under-the-hood/blob/main/gpt_training_module/notebooks/text-decoding-essay.ipynb).
8. Generate new tokens freely without spending $$$ on training.

### Step 1: Install dependencies to download OpenAI trained `GPT-2` model weights.

Originally OpenAI saved the GPT-2 parameters (weights & biases) using TensorFlow. 

So, we have to install `tensorflow` dependency to load the weights in our pytorch implementation. 

Downloading the parameters (weights & biases) may take some time, so let's install a progress bar tool called `tqdm` to track the download process.

In [1]:
pip install tensorflow>=2.15.0  tqdm>=4.66

zsh:1: 2.15.0 not found
Note: you may need to restart the kernel to use updated packages.


### Step 2: Download OpenAI trained `GPT-2` model weights.

In [2]:
import urllib.request

# Following code will download a python script which manages the whole download.
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch05/01_main-chapter-code/gpt_download.py"
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

# Import the function from downloaded script file `gpt_download.py`
from gpt_download import download_and_load_gpt2

# Key Step: Actually download loads the gpt-2 model parameters in python dictionary data structure `params`.
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


### Step 3: Write a mapping layer from `params` dictionary to the appropriate `GPTModel` components.

In [3]:
import numpy as np

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, "
                          "Right: {right.shape}"
        )

    # `torch.nn.Parameter` has two properties `weight` & `bias`.
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for trf_block in range(len(params["blocks"])):

        # Splitting one giant matrix into three smaller attention's weight matrices q_w, k_w, v_w.
        q_w, k_w, v_w = np.split((params["blocks"][trf_block]["attn"]["c_attn"])["w"], 3, axis=-1)

        # Splitting one giant matrix into three smaller attention's biases matrices q_b, k_b, v_b.
        q_b, k_b, v_b = np.split((params["blocks"][trf_block]["attn"]["c_attn"])["b"], 3, axis=-1)

        # Loading values of `W_query`, `W_key`, `W_value` in each transformer block's attention mechanism's `nn.Embedding` layer's weight parameter.
        gpt.trf_blocks[trf_block].att.W_query.weight = assign(gpt.trf_blocks[trf_block].att.W_query.weight, q_w.T)
        gpt.trf_blocks[trf_block].att.W_key.weight = assign(gpt.trf_blocks[trf_block].att.W_key.weight, k_w.T)
        gpt.trf_blocks[trf_block].att.W_value.weight = assign(gpt.trf_blocks[trf_block].att.W_value.weight, v_w.T)

        # Loading values of `W_bias`, `W_bias`, `W_bias` in each transformer block's attention mechanism's `nn.Embedding` layer's weight parameter.
        gpt.trf_blocks[trf_block].att.W_query.bias = assign(gpt.trf_blocks[trf_block].att.W_query.bias, q_b)
        gpt.trf_blocks[trf_block].att.W_key.bias = assign(gpt.trf_blocks[trf_block].att.W_key.bias, k_b)
        gpt.trf_blocks[trf_block].att.W_value.bias = assign(gpt.trf_blocks[trf_block].att.W_value.bias, v_b)

        # Loading values of `att.out_proj.weight` and `att.out_proj.bias` in each transformer block's attention mechanism.
        gpt.trf_blocks[trf_block].att.out_proj.weight = assign(gpt.trf_blocks[trf_block].att.out_proj.weight, params["blocks"][trf_block]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[trf_block].att.out_proj.bias = assign(gpt.trf_blocks[trf_block].att.out_proj.bias, params["blocks"][trf_block]["attn"]["c_proj"]["b"])

        # Loading values of weights & bias of first layer of feedforward module in each transformer block.
        gpt.trf_blocks[trf_block].ff.layers[0].weight = assign(gpt.trf_blocks[trf_block].ff.layers[0].weight, params["blocks"][trf_block]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[trf_block].ff.layers[0].bias = assign(gpt.trf_blocks[trf_block].ff.layers[0].bias, params["blocks"][trf_block]["mlp"]["c_fc"]["b"])

        # Loading values of weights & bias of second layer of feedforward module in each transformer block.
        gpt.trf_blocks[trf_block].ff.layers[2].weight = assign(gpt.trf_blocks[trf_block].ff.layers[2].weight, params["blocks"][trf_block]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[trf_block].ff.layers[2].bias = assign(gpt.trf_blocks[trf_block].ff.layers[2].bias, params["blocks"][trf_block]["mlp"]["c_proj"]["b"])

        # Loading values of scale & shift of first normalization layer in each transformer block.
        gpt.trf_blocks[trf_block].norm1.scale = assign(gpt.trf_blocks[trf_block].norm1.scale, params["blocks"][trf_block]["ln_1"]["g"])
        gpt.trf_blocks[trf_block].norm1.shift = assign(gpt.trf_blocks[trf_block].norm1.shift, params["blocks"][trf_block]["ln_1"]["b"])

        # Loading values of scale & shift of second normalization layer in each transformer block.
        gpt.trf_blocks[trf_block].norm2.scale = assign(gpt.trf_blocks[trf_block].norm2.scale, params["blocks"][trf_block]["ln_2"]["g"])
        gpt.trf_blocks[trf_block].norm2.shift = assign(gpt.trf_blocks[trf_block].norm2.shift, params["blocks"][trf_block]["ln_2"]["b"])

    # Loading values of scale & shift of normalization layer in GPT model after transformer blocks.
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])

    # Loading values of output layer (projecting embeddings to vocabulary dimensions) in GPT model.
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

### Step 4: Import our already codified `GPTModel` implementation.

In [4]:
import torch
import torch.nn as nn
import tiktoken

# Import already prepared `GPTModel` abstraction
import sys 
sys.path.append("../..")
from gpt_module import GPTModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base configuration for GPT-2 model
GPT_CONFIG_124M = {
    "vocab_size": 50257,          # Vocabulary size, the number of unique tokens available to be generated.
    "context_window_size": 1024,  # Context window size, the maximum number of tokens model can see, learn and effectively predict on.
    "emb_dim": 768,               # Embedding dimension to capture meaning in a vector space.
    "n_heads": 12,                # Number of attention heads.
    "n_layers": 12,               # Number of transformer block layers. 
    "drop_rate": 0.1,             # Dropout rate to avoid overfitting.
    "qkv_bias": False             # Weather Query-Key-Value matrices includes bias along with weights.
}

# Various size variations of `GPT-2` model.
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Create configuration from base config for the selected model variation.
selected_model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[selected_model_name])
NEW_CONFIG.update({"context_length": 1024})
NEW_CONFIG.update({"qkv_bias": True})

### Step 5: Initialize our `GPTModel` implementation (by default with random weights).

In [5]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval();

### Step 6: Load OpenAI trained `GPT-2` model parameters from OpenAI into our very own implementation `GPTModel`.

In [6]:
# Key Step: Actually loading the `params` into our implementtion.
load_weights_into_gpt(gpt, params)

# Transfer the model the GPU device.
gpt.to(device);

## Step 7: Import sophisticated text generation app function which we discussed in the last [computational essay](https://github.com/umairkhancis/llm-under-the-hood/blob/main/gpt_training_module/notebooks/text-decoding-essay.ipynb).

In [7]:
# Import already prepared `GPTModel` abstraction
import sys 
sys.path.append("../..")
from gpt_training_module import text_generation_app

## Step 8: Generate new tokens freely without spending $$$ on training.

In [9]:
# Import already prepared helper functions
from embeddings_module import text_to_token_ids, token_ids_to_text

torch.manual_seed(123)
context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
tokenized_context = tokenizer.encode(context)
input_sequence = torch.tensor(tokenized_context).unsqueeze(0)

next_token_ids = text_generation_app(
    model=gpt,
    input_text=text_to_token_ids(context, tokenizer),
    max_new_tokens=10,
    context_window_size=GPT_CONFIG_124M["context_window_size"]
)
print("openai-trained-umair-gpt>", token_ids_to_text(next_token_ids, tokenizer))

openai-trained-umair-gpt> Every effort moves you forward.

The first step is to understand


**Note:** _Finally we are able to generate our longing desired token `"forward"` given the input sequence `"Every effort moves you forward"`._ 

It is due to the fact that weights used inside our implementation are the weights of `GPT-2` model trained by OpenAI.

## What Did We Learn?

In this computational essay, we successfully leveraged OpenAI's training of GPT-2 model to power our own `GPTModel` implementation without spending thousands of dollars.

We learned:

* Why training a model should be leveraged as even a relatively small language model requires significant computational resources and financial investment.
* How to download the pretrained `GPT-2` parameters released by OpenAI.
* How pretrained model parameters are organized as weights and biases for every layer of the neural network.
* How to write a mapping layer that correctly transfers these parameters into the corresponding components of our own `GPTModel` implementation.
* How to initialize our model with random parameters and then replace them with OpenAI's pretrained parameters.
* How our previously implemented text generation application can immediately generate high-quality text once supplied with pretrained parameters.

The most important insight is that a language model consists of two independent parts:

1. **The architecture**, which defines how information flows through the network.
2. **The parameters (weights and biases)**, which encode everything the model has learned during training.

By implementing the architecture ourselves and loading OpenAI's pretrained parameters, we proved that these two concerns are completely independent. 

The same `GPTModel` implementation that previously generated incoherent text instantly became a capable language model after replacing its randomly initialized parameters with expensively and rigorously learned parameters of `GPT-2` model.

At this point, we have completed the journey of reconstructing GPT from first principles—from tokenization and attention mechanisms to training, decoding, and finally running our own implementation using OpenAI's pretrained GPT-2 parameters.